# Equity Sentiment NLP Trading Signal for UK Bank Stocks

**Domain:** Finance / NLP / Market Analytics  
**Primary Evaluation Focus:** Strategy Return, Signal Quality  
**Dataset:** Financial headlines and market prices

##  Executive Summary

This project is designed as an  portfolio case study: it does not simply run code, it explains the business problem, the modelling approach, the risks, the interpretation, and the practical deployment value.

The notebook is written for a hiring manager, recruiter, or technical interviewer to understand:

- What business problem is being solved
- Why the methodology is appropriate
- Which modelling risks were considered
- How the output could support real decisions
- How the project could be extended into production

## Key Findings

- Financial text can be converted into structured sentiment signals.
- NLP trading signals need careful backtesting to avoid overfitting and look-ahead bias.
- Sentiment is more useful as a supporting feature than a standalone trading system.
- Banking stocks are sensitive to earnings, dividends, rates and credit risk language.
- The project demonstrates practical NLP applied to financial decision support.

## Business Recommendations

- Use time-aware backtesting and avoid using future prices in signal generation.
- Combine sentiment with technical and macroeconomic features.
- Track performance by stock, sector and market regime.
- Use FinBERT-style models for finance-specific language where available.
- Include transaction costs before claiming trading profitability.

## 

This  places emphasis on:

- Clear British-English commentary
- Business-first framing
- Modelling trade-offs
- Explainability and stakeholder trust
- Production and deployment awareness
- Interview-ready explanations

# Methodology and Modelling Rationale

This section contains the original analytical workflow, upgraded with professional portfolio commentary.

The focus of the project is not only to produce a metric, but to show sound judgement. In a commercial data role, the strongest candidates are able to explain:

- Why a metric was selected
- How model risk was reduced
- What assumptions were made
- How the output supports stakeholders
- What would need to happen before production deployment

For this project, the primary evaluation focus is: **Strategy Return, Signal Quality**.

# Project 05 - Equity Sentiment: NLP Trading Signal
**Domain:** Finance / Data Analysis

NLP sentiment pipeline on financial news, generates trading signals for UK bank stocks, backtested vs Buy & Hold.

**Requirements:** `pip install transformers torch yfinance pandas matplotlib textblob`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Try FinBERT first, fall back to TextBlob
try:
    from transformers import pipeline as hf_pipeline
    finbert = hf_pipeline('text-classification', model='ProsusAI/finbert', return_all_scores=True)
    USE_FINBERT = True
    print("FinBERT loaded")
except:
    from textblob import TextBlob
    USE_FINBERT = False
    print("Using TextBlob as sentiment fallback (pip install transformers torch for FinBERT)")

def get_sentiment(text):
    if USE_FINBERT:
        scores = {r['label']: r['score'] for r in finbert(text[:512])[0]}
        return scores.get('positive',0) - scores.get('negative',0)
    else:
        return TextBlob(text).sentiment.polarity

## 1. Financial Headlines

In [ ]:
headlines = [
    ("2023-01-16","LLOY.L","Lloyds Bank reports strong Q4 profit beating analyst expectations"),
    ("2023-02-06","LLOY.L","Lloyds raises dividend as mortgage book grows steadily"),
    ("2023-03-13","LLOY.L","UK housing slowdown hits Lloyds lending volumes"),
    ("2023-04-03","LLOY.L","Lloyds posts record annual profit of 7.5 billion"),
    ("2023-05-01","LLOY.L","Lloyds CEO warns of rising bad debt provisions"),
    ("2023-06-12","LLOY.L","Lloyds launches new digital banking app"),
    ("2023-07-24","LLOY.L","Regulatory fines weigh on Lloyds quarterly results"),
    ("2023-08-14","LLOY.L","Lloyds mortgage approvals surge as rate expectations ease"),
    ("2023-09-11","LLOY.L","Lloyds announces 2bn share buyback programme"),
    ("2023-10-23","LLOY.L","Lloyds credit card defaults rise as consumers struggle"),
    ("2023-11-13","BARC.L","Barclays investment banking revenues surge in volatile markets"),
    ("2023-12-04","BARC.L","Barclays raises full-year guidance after strong Q1"),
    ("2024-01-08","BARC.L","Barclays faces US regulatory probe over securities trading"),
    ("2024-01-22","BARC.L","Barclays cuts 3000 jobs as it restructures investment bank"),
    ("2024-02-05","BARC.L","Barclays posts surprise profit miss as trading income drops"),
    ("2024-02-19","BARC.L","Barclays rated Buy by three major brokers after results"),
    ("2024-03-04","NWG.L","NatWest returns to profit as legacy issues resolved"),
    ("2024-03-18","NWG.L","NatWest government stake sale accelerates ahead of schedule"),
    ("2024-04-01","NWG.L","NatWest hits capital return target ahead of schedule"),
    ("2024-04-15","NWG.L","NatWest announces major branch closure programme"),
]

df_news = pd.DataFrame(headlines, columns=['date','ticker','headline'])
df_news['date'] = pd.to_datetime(df_news['date'])
df_news['sentiment'] = df_news['headline'].apply(get_sentiment)
df_news['signal'] = (df_news['sentiment'] > 0.1).astype(int) - (df_news['sentiment'] < -0.1).astype(int)

print(df_news[['date','ticker','sentiment','signal']].to_string())

## 2. Download Price Data & Generate Signals

In [ ]:
tickers = ['LLOY.L','BARC.L','NWG.L']
prices = {}
for t in tickers:
    try:
        p = yf.download(t, start='2022-12-01', end='2024-06-01', progress=False)
        if not p.empty:
            prices[t] = p['Close'].rename(t)
            print(f"{t}: {len(p)} trading days")
    except Exception as e:
        print(f"{t}: download failed ({e})")

if prices:
    price_df = pd.concat(prices.values(), axis=1)
    print(f"\nPrice data shape: {price_df.shape}")
    price_df.head()

## 3. Visualise Sentiment Over Time

In [ ]:
fig, axes = plt.subplots(len(df_news['ticker'].unique()), 1, figsize=(14, 10))
for i, (ticker, grp) in enumerate(df_news.groupby('ticker')):
    ax = axes[i] if len(df_news['ticker'].unique()) > 1 else axes
    colors = ['#0F1C2E' if s > 0.1 else '#C9A96E' if s < -0.1 else '#6B7A8D'
              for s in grp['sentiment']]
    ax.bar(grp['date'], grp['sentiment'], color=colors, alpha=0.8)
    ax.axhline(0, color='gray', linestyle='--', lw=0.8)
    ax.set_title(f'{ticker} Sentiment Score', fontweight='bold')
    ax.set_ylabel('Sentiment')

plt.suptitle('Equity Sentiment Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Backtest — Sentiment vs Buy & Hold

In [ ]:
if prices and 'LLOY.L' in prices:
    p = price_df[['LLOY.L']].dropna().copy()
    p.columns = ['price']
    p['return'] = p['price'].pct_change()

    lloy_news = df_news[df_news['ticker']=='LLOY.L'].set_index('date')['signal']
    p['signal'] = lloy_news.reindex(p.index, method='ffill').fillna(0)

    p['strategy_return'] = p['return'] * p['signal'].shift(1)
    p['cum_strategy'] = (1 + p['strategy_return'].fillna(0)).cumprod()
    p['cum_bah']      = (1 + p['return'].fillna(0)).cumprod()

    plt.figure(figsize=(14, 6))
    plt.plot(p.index, p['cum_strategy'], color='#C9A96E', lw=2, label='Sentiment Strategy')
    plt.plot(p.index, p['cum_bah'], color='#0F1C2E', lw=2, label='Buy & Hold')
    plt.title('LLOY.L: Sentiment Strategy vs Buy & Hold', fontweight='bold', fontsize=14)
    plt.ylabel('Cumulative Return (rebased to 1)')
    plt.legend()
    plt.tight_layout()
    plt.show()

    strat = p['cum_strategy'].iloc[-1] - 1
    bah   = p['cum_bah'].iloc[-1] - 1
    alpha = strat - bah
    print(f"Sentiment Strategy: {strat:.1%}")
    print(f"Buy & Hold:         {bah:.1%}")
    print(f"Alpha:              {alpha:.1%}")
else:
    print("Run price download cell first, or check ticker symbols for your region")

# Final Business Interpretation

## Key Findings

- Financial text can be converted into structured sentiment signals.
- NLP trading signals need careful backtesting to avoid overfitting and look-ahead bias.
- Sentiment is more useful as a supporting feature than a standalone trading system.
- Banking stocks are sensitive to earnings, dividends, rates and credit risk language.
- The project demonstrates practical NLP applied to financial decision support.

## Recommended Next Steps

- Use time-aware backtesting and avoid using future prices in signal generation.
- Combine sentiment with technical and macroeconomic features.
- Track performance by stock, sector and market regime.
- Use FinBERT-style models for finance-specific language where available.
- Include transaction costs before claiming trading profitability.

## Interview Talking Point

A strong way to discuss this project in an interview:

> "Created an NLP pipeline converting financial headlines into sentiment-based trading signals for UK bank equities, with backtesting and market-aware interpretation."

## Production Considerations

Before deploying this solution in a real business environment, I would consider:

- Data quality monitoring
- Model drift monitoring
- Clear train/test or time-based validation strategy
- Threshold tuning aligned to business cost
- Explainability for stakeholder trust
- Documentation for auditability
- A reproducible pipeline for retraining